# Text Embedding Model Fine-Tuning with NeMo AutoModel

A step-by-step tutorial for fine-tuning text embedding models using [NeMo AutoModel](https://github.com/NVIDIA-NeMo/Automodel).

We will fine-tune [**Ministral-3-3B-Instruct-2512-BF16**](https://huggingface.co/mistralai/Ministral-3-3B-Instruct-2512-BF16) as a **dense embedding bi-encoder** for semantic search, document retrieval, and retrieval-augmented generation (RAG).

> **Note:** For a complete pipeline covering synthetic data generation, hard negative mining, training, and deployment, see the [Nemotron Embed pipeline](https://github.com/NVIDIA-NeMo/Nemotron/tree/main/docs/nemotron/embed).

## Learning Goals & Architectural Concepts

Text embedding models map variable-length text into fixed-dimensional dense vectors where semantic similarity corresponds to geometric proximity (such as high cosine similarity).

### 1. Bi-Encoder Architecture & Dense Retrieval

A bi-encoder embeds queries and candidate documents into dense vectors independently:

$$\mathbf{u} = f(q), \quad \mathbf{v} = g(p)$$

Because query and document representations are decoupled, document embeddings can be precomputed and indexed offline for fast approximate nearest neighbor (ANN) retrieval. Relevance scoring is computed directly via vector similarity (such as dot product or cosine similarity) in the shared embedding space.

### 2. Contrastive Learning & the InfoNCE Objective

Bi-encoders are trained using the **InfoNCE contrastive loss**. Given a query $q_i$, its positive passage $p_i^+$, and a set of negative passages $\{p_{i,j}^-\}$, the loss maximizes the agreement between the positive pair relative to all negatives:

$$\mathcal{L}_i = -\log \frac{\exp(\text{sim}(\mathbf{q}_i, \mathbf{p}_i^+) / \tau)}{\exp(\text{sim}(\mathbf{q}_i, \mathbf{p}_i^+) / \tau) + \sum_{j} \exp(\text{sim}(\mathbf{q}_i, \mathbf{p}_{i,j}^-) / \tau)}$$

- **Temperature ($\tau$)**: Divides similarity scores by $\tau$ before the softmax. Smaller values sharpen the distribution and concentrate the negative-side gradient on the hardest negatives, encouraging stronger discrimination but increasing sensitivity to false negatives. Larger values soften the distribution and spread the negative-side gradient more evenly across negatives.
- **In-Batch Negatives**: On a single GPU, each query is scored against all passages in the local mini-batch ($B \times N$ candidates, where $B$ is local batch size and $N$ is passages per query). These candidates include one positive, $N - 1$ explicit negatives, and $(B - 1) \times N$ in-batch negatives contributed by the other samples. With $B = 4$ and $N = 2$ in this configuration, each query is evaluated against $7$ total negatives per mini-batch.
- **Hard Negatives**: Specially mined non-relevant passages that share surface or lexical keywords with the query. Mined hard negatives force the model to learn deep conceptual semantics rather than relying on trivial lexical mismatches.

### 3. Bidirectional Attention & Language Tower Extraction for Ministral-3

Standard autoregressive models use **causal attention masks** to prevent tokens from attending to future positions. For embedding representations, every token should attend to the full sequence context.

`NeMoAutoModelBiEncoder` loads `mistralai/Ministral-3-3B-Instruct-2512-BF16` with `extract_submodel: language_model` to extract the text backbone (`Ministral3Model`) and configures bidirectional attention (`is_causal: false`).

### 4. Pooling Strategies & Normalization

- **Mean (Average) Pooling**: Averages token embeddings across the sequence length, weighted by the attention mask. This is the default choice (`pooling: avg`) for embedding backbones.
- **L2 Normalization**: Every output embedding $\mathbf{v}$ is normalized such that $\|\mathbf{v}\|_2 = 1.0$. This ensures cosine similarity simplifies directly to an inner dot product:

$$\cos(\mathbf{u}, \mathbf{v}) = \frac{\mathbf{u} \cdot \mathbf{v}}{\|\mathbf{u}\|_2 \|\mathbf{v}\|_2} = \mathbf{u} \cdot \mathbf{v}$$

---
## Prerequisites & Setup

* **Hardware**: 1 or more NVIDIA GPUs with at least 80 GB VRAM (e.g. NVIDIA H100 80GB, A100 80GB) to accommodate full-parameter fine-tuning with FP32 optimizer master weights.
* **Software**: NeMo AutoModel installed (`pip install -e .` or container environment).
* **Hugging Face Access**: Access to [`mistralai/Ministral-3-3B-Instruct-2512-BF16`](https://huggingface.co/mistralai/Ministral-3-3B-Instruct-2512-BF16).

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

---
## Step 1. Prepare Retrieval Training Data

AutoModel supports JSONL retrieval datasets where each line contains a JSON object with:
- `question`: The input query string.
- `pos_doc`: A list containing the ground-truth relevant document(s).
- `neg_doc`: A list containing one or more negative / non-relevant documents.

> **Dataset Provenance:** The 32 sample query-document pairs in `sample_retrieval_data.jsonl` are sourced from the [FEVER](https://huggingface.co/datasets/nvidia/embed-nemotron-dataset-v1) subset of NVIDIA's `embed-nemotron-dataset-v1` collection (licensed CC BY-SA 4.0) to provide a lightweight local format example. For full-scale training, specify `hf://nvidia/embed-nemotron-dataset-v1/FEVER` directly in `data_dir_list` as demonstrated in `examples/retrieval/bi_encoder/ministral3_3b_instruct.yaml`.

Let us inspect the bundled sample retrieval dataset:

In [ ]:
import json
from pathlib import Path

# Resolve dataset path whether run from repository root or tutorials/embedding-biencoder/
data_path = Path("sample_retrieval_data.jsonl")
if not data_path.exists():
    data_path = Path("tutorials/embedding-biencoder/sample_retrieval_data.jsonl")

with open(data_path, "r", encoding="utf-8") as f:
    samples = [json.loads(line) for line in f]

print(f"Loaded {len(samples)} sample retrieval pairs.\n")
first = samples[0]
print("Query:", first["question"])
print("Positive Passage:", first["pos_doc"][0][:120], "...")
print("Negative Passage:", first["neg_doc"][0][:120], "...")

### Hard Negative Mining

To train competitive retrieval models, datasets should include **mined hard negatives**. AutoModel provides a mining tool at `examples/retrieval/data_utils/mine_hard_negatives.py`:

```bash
python examples/retrieval/data_utils/mine_hard_negatives.py \
    --config examples/retrieval/data_utils/mining_config.yaml \
    --mining.model_name_or_path nvidia/Nemotron-3-Embed-8B-BF16 \
    --mining.train_qa_file_path raw_qa_pairs.json \
    --mining.train_file_output_path mined_train_data.json \
    --mining.hard_neg_margin 0.95
```

This indexes the corpus using an initial embedding model (such as [`nvidia/Nemotron-3-Embed-8B-BF16`](https://huggingface.co/nvidia/Nemotron-3-Embed-8B-BF16)), retrieves top candidates for each query, and filters out false negatives using the `hard_neg_margin` threshold.

---
## Step 2. Configure the Bi-Encoder Training Recipe

AutoModel uses declarative YAML recipes to configure the model, dataset, optimizer, and training schedule.
Let us inspect the configuration in `ministral3_3b_embed_config.yaml` (see also the full recipe at `examples/retrieval/bi_encoder/ministral3_3b_instruct.yaml`):

In [ ]:
from pathlib import Path

# Locate recipe config whether run from repository root or tutorials/embedding-biencoder/
cfg_path = Path("ministral3_3b_embed_config.yaml")
if not cfg_path.exists():
    cfg_path = Path("tutorials/embedding-biencoder/ministral3_3b_embed_config.yaml")

with open(cfg_path, "r", encoding="utf-8") as f:
    print(f.read())

### Key Configuration Parameters Explained

1. **`recipe: TrainBiEncoderRecipe`**: Selects AutoModel's dedicated contrastive bi-encoder training pipeline with InfoNCE loss and in-batch negative support.
2. **`temperature: 0.02`**: Divides cosine similarities by 0.02 before the softmax, sharpening the probability distribution and making the loss more sensitive to the highest-scoring (hard) negatives.
3. **`model._target_: nemo_automodel.NeMoAutoModelBiEncoder.from_pretrained`**: Loads `mistralai/Ministral-3-3B-Instruct-2512-BF16` with `extract_submodel: language_model`, `is_causal: false`, average pooling (`pooling: avg`), and L2 normalization (`l2_normalize: true`).
4. **`optimizer._target_: transformer_engine.pytorch.optimizers.fused_adam.FusedAdam`**: Configured with `master_weights: true` and `adam_w_mode: true` to preserve FP32 master weights during BF16 training so small updates do not round away.
5. **`step_scheduler.save_checkpoint_every_epoch: false`**: Prevents saving full checkpoints on every epoch with small tutorial datasets, preserving the intended step interval (`ckpt_every_steps: 10`). Since no validation dataset is configured for this tutorial, `val_every_steps` is also commented out in `step_scheduler`.
6. **`save_consolidated: final`**: Consolidates sharded weights into a Hugging Face compatible `model.safetensors` export upon training completion, avoiding repeated I/O on intermediate checkpoints.

---
## Step 3. Launching Bi-Encoder Fine-Tuning

You can launch the training job directly using the AutoModel CLI or programmatically in Python.

### CLI Launch Command

```bash
# Single-GPU training:
automodel tutorials/embedding-biencoder/ministral3_3b_embed_config.yaml --nproc-per-node 1

# Multi-GPU training with FSDP2 (with 8 DP ranks and local_batch_size 4, global_batch_size must be a multiple of 32):
automodel tutorials/embedding-biencoder/ministral3_3b_embed_config.yaml --nproc-per-node 8 --step_scheduler.global_batch_size 32
```

### Programmatic Execution in Python

In [ ]:
from pathlib import Path

from nemo_automodel.components.config._arg_parser import parse_args_and_load_config

# Locate recipe config and resolve common repository root
cfg_path = Path("ministral3_3b_embed_config.yaml")
if not cfg_path.exists():
    cfg_path = Path("tutorials/embedding-biencoder/ministral3_3b_embed_config.yaml")

tutorial_dir = cfg_path.parent.resolve()
repo_root = tutorial_dir.parent.parent if tutorial_dir.name == "embedding-biencoder" else tutorial_dir
output_dir = (repo_root / "output" / "ministral3_3b_embed_tutorial").resolve()
checkpoint_base = output_dir / "checkpoints"

cfg = parse_args_and_load_config(str(cfg_path))

# Ensure dataset and checkpoint paths resolve to common absolute directories
data_path = tutorial_dir / "sample_retrieval_data.jsonl"
cfg.dataset.data_dir_list = [str(data_path.resolve())]
cfg.checkpoint.checkpoint_dir = str(checkpoint_base)

print(f"Configured recipe: {cfg.recipe}")
print(f"Model backbone:    {cfg.model.pretrained_model_name_or_path}")
print(f"Extract submodel:  {cfg.model.extract_submodel}")
print(f"In-batch negative: {cfg.model.do_distributed_inbatch_negative}")
print(f"Temperature:       {cfg.temperature}")
print(f"Pooling:           {cfg.model.pooling}")
print(f"L2 Normalization:  {cfg.model.l2_normalize}")
print(f"Checkpoint root:   {cfg.checkpoint.checkpoint_dir}")

# To execute training, uncomment:
# recipe = TrainBiEncoderRecipe(cfg)
# recipe.setup()
# recipe.run_train_validation_loop()

### What to Monitor During Training

This 20-step tutorial run is intended to verify that the training pipeline works, not to demonstrate model convergence. Exact metric values depend on the hardware, batch configuration, random seed, and input data.

During training, verify that:
- `loss` and `grad_norm` remain finite. Loss may fluctuate because the tutorial repeatedly trains on a small dataset.
- The learning rate warms up for the configured two steps and then follows linear decay.
- Periodic and final checkpoints are created successfully.
- Training completes without out-of-memory or distributed communication errors.

---
## Step 4. Inference, Semantic Search, & Verification

Once training completes, we can load the Ministral-3-3B bi-encoder and verify that query embeddings match their corresponding positive documents.

In [ ]:
from pathlib import Path

import torch

from nemo_automodel import NeMoAutoModelBiEncoder, NeMoAutoTokenizer
from nemo_automodel.components.checkpoint.utils import find_latest_checkpoint

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Resolve common absolute output root across CLI and notebook execution
current_dir = Path.cwd().resolve()
repo_root = current_dir if (current_dir / "tutorials" / "embedding-biencoder").exists() else current_dir.parent.parent
checkpoint_base = (repo_root / "output" / "ministral3_3b_embed_tutorial" / "checkpoints").resolve()

# Select the latest completed checkpoint numerically
latest_ckpt = find_latest_checkpoint(checkpoint_base)
if latest_ckpt is None:
    raise FileNotFoundError(
        f"No completed checkpoints found under {checkpoint_base}. "
        "Please complete Step 3 training before running verification."
    )

consolidated_candidate = Path(latest_ckpt) / "model" / "consolidated"
if not consolidated_candidate.exists() or not (consolidated_candidate / "config.json").exists():
    raise FileNotFoundError(
        f"Consolidated checkpoint directory not found at {consolidated_candidate}. "
        "Ensure save_consolidated: final is enabled and training has completed."
    )

eval_model_path = str(consolidated_candidate)
print(f"Found fine-tuned consolidated checkpoint: {eval_model_path}")

# Load tokenizer from consolidated export
tokenizer = NeMoAutoTokenizer.from_pretrained(
    eval_model_path,
    padding_side="left",
    tokenizer_backend="tokenizers",
    fix_mistral_regex=True,
    split_special_tokens=True,
    add_bos_token=True,
    add_eos_token=False,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load bi-encoder: consolidated checkpoint is already the extracted text backbone with bidirectional attention
print(f"Loading {eval_model_path} as NeMoAutoModelBiEncoder (pooling=avg, l2_norm=True)...")
model = NeMoAutoModelBiEncoder.from_pretrained(
    eval_model_path,
    trust_remote_code=True,
    pooling="avg",
    l2_normalize=True,
    is_causal=False,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
).to(device)
model.eval()
print("Bi-Encoder successfully loaded!")

### Computing Similarity Across Queries and Passages

In [ ]:
queries = [
    "query: Did David Harbour perform in theater?",
    "query: Which comic book character was the first black superhero in mainstream comics?",
]

passages = [
    "passage: David Harbour ( born April 10 , 1974 ) is an American actor who has performed in film , television , and the theatre .",
    "passage: The Black Panther ( T'Challa ) is a fictional superhero appearing in American comic books published by Marvel Comics . He is the first black superhero in mainstream American comics , debuting years before early African-American superheroes such as the Falcon and Luke Cage .",
    "passage: The Curtis Institute of Music is a conservatory in Philadelphia that offers courses of study leading to a performance diploma .",
]

# Tokenize inputs
q_inputs = tokenizer(queries, padding=True, truncation=True, return_tensors="pt").to(device)
p_inputs = tokenizer(passages, padding=True, truncation=True, return_tensors="pt").to(device)

# Generate normalized embeddings using bi-encoder mapping-in, tensor-out API
with torch.no_grad():
    q_embeds = model(q_inputs)  # [N_queries, D]
    p_embeds = model(p_inputs)  # [N_passages, D]

# Compute cosine similarity matrix (inner dot product of unit-length vectors)
sim_matrix = torch.matmul(q_embeds, p_embeds.T)

print("Cosine Similarity Matrix:")
for i, q in enumerate(queries):
    print(f"\nQuery {i + 1}: {q}")
    best_idx = torch.argmax(sim_matrix[i]).item()
    for j, p in enumerate(passages):
        marker = " <-- [Top Match]" if j == best_idx else ""
        print(f"  Passage {j + 1} score: {sim_matrix[i, j]:.4f}{marker}")

---
## Step 5. Export & Production Deployment

Because `save_consolidated: final` is configured in `ministral3_3b_embed_config.yaml`, the checkpointer exports Hugging Face compatible artifacts into the consolidated directory of the completed run (`<checkpoint_dir>/<latest_checkpoint>/model/consolidated/`):

- `model.safetensors`
- `config.json`
- `tokenizer.json` / `tokenizer_config.json`

You can load this exported checkpoint in downstream RAG frameworks by reusing the export path resolved in Step 4:

```python
from sentence_transformers import SentenceTransformer

# Load fine-tuned bi-encoder directly from the resolved consolidated export
embedder = SentenceTransformer(eval_model_path)
embeddings = embedder.encode(["Hello world", "Semantic search with NeMo AutoModel"])
print("Embeddings shape:", embeddings.shape)
```

## Summary

In this tutorial, you learned:
1. How bi-encoders decouple query and passage encoding for scalable semantic search.
2. The mathematics behind contrastive InfoNCE loss, temperature scaling, and in-batch / hard negatives.
3. How `NeMoAutoModelBiEncoder` extracts the language tower (`extract_submodel: language_model`) of `Ministral-3-3B-Instruct-2512-BF16` and configures bidirectional attention (`is_causal: false`) with mean pooling.
4. How to configure and launch training with `TrainBiEncoderRecipe`.
5. How to verify cosine similarity rankings and export consolidated checkpoints for downstream retrieval pipelines.